In [2]:
import pandas as pd
from pymongo import MongoClient
from joblib import Parallel, delayed
from tqdm import tqdm

config_files = [
    "pom.xml",
    "requirements.txt",
    "setup.py",
    "pyproject.toml",
    "setup.cfg",
]
client = MongoClient("127.0.0.1", 27017)
db = client["bridge"]

In [ ]:
def count_candidate_update():
    num_commits = []
    num_blobs = []
    py_col = db["py_candidate_update_commits"]
    java_col = db["java_candidate_update_commits"]

    java_df = pd.DataFrame(java_col.find({}, projection={"_id": 0, "filepath": 0}))
    num_commits.append(java_df["commit"].nunique())
    num_blobs.append(pd.concat([java_df["new_blob"], java_df["old_blob"]]).nunique())
    del java_df

    def single_pydoc_process(doc):
        filepath = doc["filepath"].split("/")[-1]
        return [
            config_files.index(filepath),
            doc["commit"],
            doc["new_blob"],
            doc["old_blob"],
        ]

    py_results = Parallel(n_jobs=100)(
        delayed(single_pydoc_process)(doc)
        for doc in tqdm(
            py_col.find({}, projection={"_id": 0}),
            total=py_col.estimated_document_count(),
        )
    )

    tmp_py_commits = {i: set() for i in range(1, 5)}
    tmp_py_blobs = {i: set() for i in range(1, 5)}
    for idx, cmt, new_blob, old_blob in py_results:
        tmp_py_commits[idx].add(cmt)
        tmp_py_blobs[idx].add(new_blob)
        tmp_py_blobs[idx].add(old_blob)
    total_blobs = set()
    total_commits = set()
    for i in range(1, 5):
        total_commits |= tmp_py_commits[i]
        total_blobs |= tmp_py_blobs[i]
        num_commits.append(len(tmp_py_commits[i]))
        num_blobs.append(len(tmp_py_blobs[i]))
    num_commits.append(len(total_commits))
    num_blobs.append(len(total_blobs))
    return {"# CU Commits": num_commits, "# CFG Blobs": num_blobs}


data1 = count_candidate_update()
data1

100%|██████████| 29979360/29979360 [06:53<00:00, 72504.83it/s] 


{'# CU Commits': [31852062, 14677782, 7622076, 2725506, 1174085, 25002699],
 '# CFG Blobs': [63959850, 10189499, 5525611, 2045404, 719294, 18479562]}

In [4]:
def count_version_bumping():
    num_commits = []
    py_col = db["py_version_bumping_commits"]
    java_col = db["java_version_bumping_commits"]
    py_df = pd.DataFrame(py_col.find({}, projection={"_id": 0}))
    java_df = pd.DataFrame(java_col.find({}, projection={"_id": 0}))
    num_commits.append(java_df["commit"].nunique())
    for f in config_files[1:]:
        tmp = py_df[py_df["filepath"].str.endswith(f)]
        num_commits.append(tmp["commit"].nunique())
    num_commits.append(py_df["commit"].nunique())
    return {"# VB Commits": num_commits}


data2 = count_version_bumping()

In [6]:
def process_single(doc):
    data = [0, 0, 0, 0]
    cfg_files = [cfg["filepath"].split("/")[-1] for cfg in doc["configuration_files"]]
    blobs = set()
    for fbb in doc["code_files"]:
        blobs.add(fbb["new_blob"])
        blobs.add(fbb["old_blob"])
    for i, f in enumerate(config_files[1:]):
        if f in cfg_files:
            data[i] = (1, blobs)
        else:
            data[i] = (0, set())
    return data


def count_update():
    num_commits = []
    num_blobs = []
    py_col = db["py_update_commits"]
    java_col = db["java_update_commits"]
    num_commits.append(java_col.estimated_document_count())
    tmp_java_blobs = set()
    for doc in tqdm(
        java_col.find({}, projection={"_id": 0}),
        total=java_col.estimated_document_count(),
    ):
        for fbb in doc["code_files"]:
            tmp_java_blobs.add(fbb["new_blob"])
            tmp_java_blobs.add(fbb["old_blob"])
    num_blobs.append(len(tmp_java_blobs))
    del tmp_java_blobs

    tmp_py_result = Parallel(n_jobs=100)(
        delayed(process_single)(doc)
        for doc in tqdm(
            py_col.find({}, projection={"_id": 0}),
            total=py_col.estimated_document_count(),
        )
    )

    total_blobs = set()
    for i in range(4):
        cmt = 0
        blobs = set()
        for res in tmp_py_result:
            cmt += res[i][0]
            blobs |= res[i][1]
        total_blobs |= blobs
        num_commits.append(cmt)
        num_blobs.append(len(blobs))

    num_commits.append(py_col.estimated_document_count())
    num_blobs.append(len(total_blobs))

    return {"# Update Commits": num_commits, "# Code Blobs": num_blobs}


data3 = count_update()

100%|██████████| 753902/753902 [05:09<00:00, 2439.50it/s]


In [11]:
pd.DataFrame({"Configuration File": config_files + ["Total"]} | data1 | data2 | data3)

,Configuration File,# CU Commits,# CFG Blobs,# VB Commits,# Update Commits,# Code Blobs
0,pom.xml,31852062,63959850,6103952,1049834,27252177
1,requirements.txt,14677782,10189499,7719071,672536,6745851
2,setup.py,7622076,5525611,238418,78275,1394187
3,pyproject.toml,2725506,2045404,64965,16586,266739
4,setup.cfg,1174085,719294,29832,7886,131717
5,Total,25002699,18479562,7993598,753902,7794556
